# DefectosCafeVerde — DVF2 selective dual-view fusion

Trains one final small selective fuser from the frozen DVF1 train cache. The D0 detector is never loaded or trained. Validation is evaluated once after epoch 100, and test is never accessed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

relative=Path('experiments/defectoscafeverde-dvf1-v1/DVF1_seed42_result.json')
roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
matches=sorted({p for root in roots if root.exists() for p in root.rglob(relative.name) if p.as_posix().endswith(relative.as_posix())})
if not matches: raise FileNotFoundError(f'DVF1 result tidak ditemukan: {relative}')
digests={sha256(path) for path in matches}
if len(digests)!=1: raise RuntimeError(f'Ditemukan DVF1 result berbeda: {matches}')
preferred=[p for p in matches if '/MyDrive/' in p.as_posix()]
DVF1_RESULT=(preferred or matches)[0]
DVF1_ROOT=DVF1_RESULT.parent
DVF1_REVIEW=DVF1_ROOT/'DVF1_seed42_review.json'
for path in (DVF1_REVIEW,DVF1_ROOT/'DVF1_last.pt',DVF1_ROOT/'train_pair_cache.pt',DVF1_ROOT/'val_pair_cache.pt'):
    if not path.is_file(): raise FileNotFoundError(path)
PROJECT=DVF1_RESULT.parents[2]
OUT=PROJECT/'experiments/defectoscafeverde-dvf2-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT)
print('DVF1 RESULT:',DVF1_RESULT)
print('DVF1 REVIEW:',DVF1_REVIEW)
print('OUTPUT:',OUT)

In [ ]:
import importlib, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
check=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dvf2.py','tests/test_defectoscafeverde_dual_view_fusion.py'],cwd=REPO,text=True,capture_output=True)
print(check.stdout)
if check.returncode: print(check.stderr); raise RuntimeError('DVF2 tests gagal sebelum training')

In [ ]:
from coffee_detector.experiments.run_defectoscafeverde_dvf2 import run_static_audit
STATIC=OUT/'static_audit.json'
audit=run_static_audit(STATIC)
print('PARAMETERS:',audit['parameters'])
print('GATES:',audit['gates'])
print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit DVF2 gagal; jangan training.'

In [ ]:
import time, torch
DEVICE='0' if torch.cuda.is_available() else 'cpu'
LOG=OUT/'DVF2_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_dvf2','--dvf1-result',str(DVF1_RESULT),'--dvf1-review',str(DVF1_REVIEW),'--output-root',str(OUT),'--device',DEVICE,'--authorize-training']
print('START/RESUME: DVF2 | device=',DEVICE,'| log=',LOG,flush=True)
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    last=''
    while process.poll() is None:
        time.sleep(15)
        lines=LOG.read_text(errors='replace').splitlines()
        status=[line for line in lines if line.startswith(('DVF2 FUSER','REUSE COMPLETE DVF2'))]
        if status and status[-1]!=last: last=status[-1]; print(last,flush=True)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'DVF2 gagal: {process.returncode}')
RESULT=OUT/'DVF2_seed42_result.json'
result=json.loads(RESULT.read_text())
assert result['test_images_accessed'] is False
print(json.dumps(result,indent=2,ensure_ascii=False))

In [ ]:
import pandas as pd
metric_names=['physical_pair_accuracy','macro_class_accuracy','bottom3_class_accuracy','worst_class_accuracy']
rows=[{'model':name,**{metric:values[metric] for metric in metric_names}} for name,values in result['values'].items()]
display(pd.DataFrame(rows).set_index('model').style.format('{:.2%}'))
dvf1=result['values']['DVF1']['accuracy_by_class']; dvf2=result['values']['DVF2']['accuracy_by_class']
class_rows=[{'class_name':name,'DVF1':dvf1[name],'DVF2':dvf2[name],'delta':dvf2[name]-dvf1[name]} for name in dvf1]
display(pd.DataFrame(class_rows).sort_values('delta',ascending=False).style.format({'DVF1':'{:.2%}','DVF2':'{:.2%}','delta':'{:+.2%}'}))
print('COMPARISON:',result['comparison'])
print('GATE:',result['gate_diagnostics'])
print('GATES:',result['gates'])
print('PARAMETERS:',result['fuser_parameters'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| FUSER TRAINING:',result['fuser_training_executed'],'| TRAINING THIS CALL:',result['training_executed_this_call'],'| TEST:',result['test_images_accessed'])